In [59]:
import pandas as pd
from pathlib import Path   
import numpy as np

In [60]:
result_path = Path("../thesis_results/downstream_task")
methods_list = ["uniform","psa", "kmm", "mrs-forest", "soft-mrs-exponential", "fw-mrs-temperature",  "fw-mrs-temperature-svm"]
# bias_types = ["less_negative_class", "less_positive_class", "mean_difference"]
bias_types = ["mean_difference"]
metrics = ["AUROC", "AUPRC"]
# less_bias_strengths = ["0.1", "0.2", "0.3"]
less_bias_strengths = ["0.1"]
mean_bias_strengthts = ["0.8"]
datasets = ["folktables_employment", "folktables_income", "hr_analytics", "breast_cancer", "loan_prediction"]

In [61]:
aurocs = []
auprcs = []
dict_list = []
for dataset in datasets:
    for bias_type in bias_types:
        for method in methods_list:
            if bias_type == "mean_difference":
                bias_strengths = mean_bias_strengthts
            else : 
                bias_strengths = less_bias_strengths
            for bias_strength in bias_strengths:
                json_file = result_path / dataset / bias_type /  bias_strength/ method / "classification_results.json"
                result_file = pd.read_json(str(json_file))
                dict_list.append(
                    {
                        "Method": method, "Data Set": dataset, 
                        "AUROC Mean": result_file["random forest auroc"]["mean"], 
                        "AUROC Std": result_file["random forest auroc"]["sd"], 
                        "AUPRC Mean": result_file["random forest auprc"]["mean"], 
                        "AUPRC Std": result_file["random forest auprc"]["sd"], 
                        "Bias Type": bias_type, "Bias Strength": bias_strength,
                        "Dropped Samples Mean": result_file["dropped_samples"]["mean"],
                        "Dropped Samples Std": result_file["dropped_samples"]["std"]
                    }
                                )
result_df = pd.DataFrame(data=dict_list)

In [62]:
result_df = result_df.replace({"uniform": "Uniform", "kmm": "KMM", "psa": "PSA", "mrs-forest": "MRS",
                               "soft-mrs-linear": "Soft-MRS-Linear", "soft-mrs-exponential": "Soft-MRS-Exponential",
                               "fw-mrs-temperature-svm": "FW-MRS-SVM", "fw-mrs-temperature": "FW-MRS-RF"})
result_df


,Method,Data Set,AUROC Mean,AUROC Std,AUPRC Mean,AUPRC Std,Bias Type,Bias Strength,Dropped Samples Mean,Dropped Samples Std
0,Uniform,folktables_employment,0.885053,0.007930,0.845045,0.013863,mean_difference,0.8,0.00,0.000000
1,PSA,folktables_employment,0.885395,0.007804,0.845636,0.014192,mean_difference,0.8,0.00,0.000000
2,KMM,folktables_employment,0.881620,0.008066,0.839454,0.015406,mean_difference,0.8,0.00,0.000000
3,MRS,folktables_employment,0.884941,0.007886,0.844889,0.014057,mean_difference,0.8,194.30,37.416707
4,Soft-MRS-Exponential,folktables_employment,0.885599,0.008035,0.845740,0.013996,mean_difference,0.8,0.00,0.000000
5,FW-MRS-RF,folktables_employment,0.876028,0.008141,0.832883,0.013800,mean_difference,0.8,188.50,42.311346
6,FW-MRS-SVM,folktables_employment,0.883324,0.008371,0.842921,0.013805,mean_difference,0.8,478.90,67.448425
7,Uniform,folktables_income,0.855276,0.008886,0.814747,0.013880,mean_difference,0.8,0.00,0.000000
8,PSA,folktables_income,0.858145,0.009160,0.818174,0.013560,mean_difference,0.8,0.00,0.000000
9,KMM,folktables_income,0.853194,0.009154,0.811642,0.014052,mean_difference,0.8,0.00,0.000000


In [63]:
for bias_type in bias_types:
    if bias_type == "mean_difference":
        bias_strengths = mean_bias_strengthts
    else : 
        bias_strengths = less_bias_strengths
    for bias_strength in bias_strengths:
        print(f"{bias_type}, {bias_strength}")
        for method in result_df["Method"].unique():
            mean_auroc_values = []
            std_auroc_values = []
            for dataset in datasets:
                mean_auroc = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                    (result_df["Bias Strength"]==bias_strength) & 
                                                    (result_df["Data Set"]==dataset)]["AUROC Mean"].iloc[0]
                mean_auroc_values.append(np.round(mean_auroc, 3))

                std_auroc = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                    (result_df["Bias Strength"]==bias_strength) & 
                                                    (result_df["Data Set"]==dataset)]["AUROC Std"].iloc[0]
                std_auroc_values.append(np.round(std_auroc, 3))

            print(f"\t& {method} \
& ${mean_auroc_values[0]}\\pm{std_auroc_values[0]}$ \
& ${mean_auroc_values[1]}\\pm{std_auroc_values[1]}$ \
& ${mean_auroc_values[2]}\\pm{std_auroc_values[2]}$ \
& ${mean_auroc_values[3]}\\pm{std_auroc_values[3]}$ \
& ${mean_auroc_values[4]}\\pm{std_auroc_values[4]}$ & \\\\")
        print("\n")

mean_difference, 0.8
	& Uniform & $0.885\pm0.008$ & $0.855\pm0.009$ & $0.765\pm0.018$ & $0.991\pm0.005$ & $0.761\pm0.054$ & \\
	& PSA & $0.885\pm0.008$ & $0.858\pm0.009$ & $0.762\pm0.018$ & $0.991\pm0.005$ & $0.756\pm0.057$ & \\
	& KMM & $0.882\pm0.008$ & $0.853\pm0.009$ & $0.756\pm0.019$ & $0.99\pm0.006$ & $0.75\pm0.055$ & \\
	& MRS & $0.885\pm0.008$ & $0.855\pm0.009$ & $0.764\pm0.018$ & $0.991\pm0.005$ & $0.755\pm0.055$ & \\
	& Soft-MRS-Exponential & $0.886\pm0.008$ & $0.859\pm0.009$ & $0.759\pm0.019$ & $0.991\pm0.005$ & $0.753\pm0.056$ & \\
	& FW-MRS-RF & $0.876\pm0.008$ & $0.851\pm0.011$ & $0.762\pm0.016$ & $0.988\pm0.008$ & $0.735\pm0.061$ & \\
	& FW-MRS-SVM & $0.883\pm0.008$ & $0.85\pm0.011$ & $0.759\pm0.018$ & $0.989\pm0.007$ & $0.737\pm0.053$ & \\




In [64]:
for bias_type in bias_types:
    if bias_type == "mean_difference":
        bias_strengths = mean_bias_strengthts
    else : 
        bias_strengths = less_bias_strengths
    for bias_strength in bias_strengths:
        print(f"{bias_type}, {bias_strength}")
        for method in result_df["Method"].unique():
            mean_auprc_values = []
            std_auprc_values = []
            for dataset in datasets:
                mean_auprc = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                    (result_df["Bias Strength"]==bias_strength) & 
                                                    (result_df["Data Set"]==dataset)]["AUPRC Mean"].iloc[0]
                mean_auprc_values.append(np.round(mean_auprc, 3))

                std_auprc = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                    (result_df["Bias Strength"]==bias_strength) & 
                                                    (result_df["Data Set"]==dataset)]["AUPRC Std"].iloc[0]
                std_auprc_values.append(np.round(std_auprc, 3))

            print(f"\t& {method} \
& ${mean_auprc_values[0]}\\pm{std_auprc_values[0]}$ \
& ${mean_auprc_values[1]}\\pm{std_auprc_values[1]}$ \
& ${mean_auprc_values[2]}\\pm{std_auprc_values[2]}$ \
& ${mean_auprc_values[3]}\\pm{std_auprc_values[3]}$ \
& ${mean_auprc_values[4]}\\pm{std_auprc_values[4]}$ & \\\\")
        print("\n")

mean_difference, 0.8
	& Uniform & $0.845\pm0.014$ & $0.815\pm0.014$ & $0.47\pm0.029$ & $0.996\pm0.002$ & $0.856\pm0.039$ & \\
	& PSA & $0.846\pm0.014$ & $0.818\pm0.014$ & $0.469\pm0.031$ & $0.996\pm0.002$ & $0.852\pm0.043$ & \\
	& KMM & $0.839\pm0.015$ & $0.812\pm0.014$ & $0.462\pm0.027$ & $0.996\pm0.002$ & $0.85\pm0.039$ & \\
	& MRS & $0.845\pm0.014$ & $0.815\pm0.014$ & $0.47\pm0.028$ & $0.996\pm0.002$ & $0.852\pm0.039$ & \\
	& Soft-MRS-Exponential & $0.846\pm0.014$ & $0.819\pm0.014$ & $0.468\pm0.03$ & $0.996\pm0.002$ & $0.848\pm0.041$ & \\
	& FW-MRS-RF & $0.833\pm0.014$ & $0.81\pm0.016$ & $0.468\pm0.028$ & $0.994\pm0.005$ & $0.833\pm0.048$ & \\
	& FW-MRS-SVM & $0.843\pm0.014$ & $0.807\pm0.014$ & $0.465\pm0.03$ & $0.995\pm0.003$ & $0.84\pm0.036$ & \\




In [65]:
result_df["Rank AUROC"] = result_df.groupby("Data Set")["AUROC Mean"].rank(ascending=False)
result_df["Rank AUPRC"] = result_df.groupby("Data Set")["AUPRC Mean"].rank(ascending=False)
result_df[["Method", "Rank AUROC", "Rank AUPRC"]].groupby("Method").mean()

,Rank AUROC,Rank AUPRC
Method,,
FW-MRS-RF,6.0,6.2
FW-MRS-SVM,6.0,6.0
KMM,5.6,5.4
MRS,3.0,2.8
PSA,2.8,2.6
Soft-MRS-Exponential,2.4,2.6
Uniform,2.2,2.4
